# API errors

**Objective:** Translate domain failures into consistent HTTP responses.

## Simple version

In [ ]:
# HTTPException is a direct way to return an HTTP error from a small route.
from fastapi import HTTPException


def get_task(task_id: int) -> dict:
    if task_id != 1:
        raise HTTPException(status_code=404, detail="Task not found")
    return {"id": 1}


try:
    get_task(99)
except HTTPException as error:
    print(error.status_code, error.detail)

## Polished version

In [ ]:
# Business code raises a domain error; the API translates it to HTTP.
import httpx
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse


class TaskNotFound(Exception):
    def __init__(self, task_id: int) -> None:
        self.task_id = task_id


class TaskService:
    def __init__(self, tasks: dict[int, dict]) -> None:
        self.tasks = tasks

    def get(self, task_id: int) -> dict:
        task = self.tasks.get(task_id)
        if task is None:
            raise TaskNotFound(task_id)
        return task


def create_app(service: TaskService) -> FastAPI:
    app = FastAPI()

    # One handler keeps the error response consistent across all routes.
    @app.exception_handler(TaskNotFound)
    async def task_not_found(
        request: Request,
        error: TaskNotFound,
    ) -> JSONResponse:
        return JSONResponse(
            status_code=404,
            content={
                "code": "task_not_found",
                "message": "Task not found",
                "task_id": error.task_id,
            },
        )

    @app.get("/tasks/{task_id}")
    async def get_task(task_id: int) -> dict:
        return service.get(task_id)

    return app


app = create_app(TaskService({1: {"id": 1, "title": "Learn errors"}}))
transport = httpx.ASGITransport(app=app)
async with httpx.AsyncClient(transport=transport, base_url="http://test") as client:
    response = await client.get("/tasks/99")

print(response.status_code, response.json())